# CNN-SRM — Cross-Database Deepfake Detection Experiment

**Research question:** Does a noise-pattern detector trained on one deepfake dataset generalize to others?

**Model overview:**  
CNN-SRM uses Steganalysis Rich Model (SRM) filters to extract noise/frequency residuals  
from images, then learns to classify them via a pre-trained autoencoder + classifier head.

**Option A — clean cross-database experiment:**  
For each training database, a **fresh autoencoder is trained from scratch** on that database's  
SRM features. This ensures zero knowledge leakage from other databases.

**Per-database pipeline:**
1. Apply SRM filters (8 filters, 5×5) → (256, 256, 24) feature maps
2. Train autoencoder on that database's SRM features (reconstruction task)
3. Extract encoder (latent: 16×16×256)
4. Phase 1 — freeze encoder, train classifier head
5. Phase 2 — unfreeze top 30% of encoder, fine-tune end-to-end
6. Evaluate on all 4 databases (within + cross)

**Databases:** OpenForensics · CustomWar · CelebDF · CiFake

In [ ]:
# =================== CELL 1: SETUP ===================
import os
import gc
import random
import pickle
import numpy as np
import tensorflow as tf
import mimetypes
from datetime import datetime

os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/nvidia-cuda-toolkit'
tf.config.optimizer.set_jit(False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('✅ GPU memory growth enabled')
    except RuntimeError as e:
        print(e)

tf.keras.backend.clear_session()
print(f'✅ Setup complete — TensorFlow {tf.__version__}')

In [ ]:
# =================== CELL 2: PATHS & TENSORBOARD CONFIG ===================

GDRIVE_PATH   = os.path.expanduser('~/RealEyes/gdrive')
DATASET_ROOT  = os.path.join(GDRIVE_PATH, 'data_set_split')
DATASETS_DIR  = os.path.expanduser('~/RealEyes/RealEyes/datasets')
CELEBDF_DIR   = os.path.join(DATASETS_DIR, 'celebdf_v2')

MODEL_NAME            = 'cnn_srm'
EXPERIMENT_MODELS_DIR = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/no_lora_experiment'  # no-LoRA run
EXPERIMENT_MODELS_DIR = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/no_lora_experiment'  # no-LoRA run
os.makedirs(MODEL_DIR, exist_ok=True)

TB_LOG_ROOT = os.path.expanduser('~/RealEyes/tensorboard_logs')
os.makedirs(TB_LOG_ROOT, exist_ok=True)


def get_tb_log_dir(train_db_name, suffix='train'):
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    return os.path.join(TB_LOG_ROOT, MODEL_NAME, f'{suffix}_{train_db_name}', ts)


def log_eval_to_tb(train_db_name, test_db_name, metrics: dict, step=0):
    log_dir = os.path.join(
        TB_LOG_ROOT, MODEL_NAME, 'cross_db_eval',
        f'train_{train_db_name}__test_{test_db_name}'
    )
    writer = tf.summary.create_file_writer(log_dir)
    with writer.as_default():
        for name, value in metrics.items():
            tf.summary.scalar(name, float(value), step=step)
    writer.flush()


print('✅ Paths ready')
print(f'  MODEL_DIR  : {MODEL_DIR}')
print(f'  TB_LOG_ROOT: {TB_LOG_ROOT}')
print()
print('▶  View TensorBoard:')
print(f'  [server ] tensorboard --logdir {TB_LOG_ROOT} --port 6006 --bind_all')
print( '  [local  ] ssh -L 6006:localhost:6006 <user>@<server_ip>')
print( '  [browser] http://localhost:6006')

In [ ]:
# =================== CELL 3: DATABASE DEFINITIONS ===================

DATABASES = {}


def _try_add(name, paths):
    missing = [k for k, v in paths.items() if not os.path.isdir(v)]
    if missing:
        print(f'  ⚠️  {name}: missing splits {missing} — skipped')
        return
    DATABASES[name] = paths
    print(f'  ✅ {name}')


print('🗂  Scanning databases...')

_try_add('OpenForensics', {
    'train': os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Train'),
    'val':   os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Validation'),
    'test':  os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Test'),
})
_try_add('CustomWar', {
    'train': os.path.join(DATASET_ROOT, 'train'),
    'val':   os.path.join(DATASET_ROOT, 'val'),
    'test':  os.path.join(DATASET_ROOT, 'test'),
})
_try_add('CelebDF', {
    'train': os.path.join(CELEBDF_DIR, 'train'),
    'val':   os.path.join(CELEBDF_DIR, 'val'),
    'test':  os.path.join(CELEBDF_DIR, 'test'),
})
_try_add('CiFake', {
    'train': os.path.join(DATASETS_DIR, 'cifake/train'),
    'val':   os.path.join(DATASETS_DIR, 'cifake/test'),
    'test':  os.path.join(DATASETS_DIR, 'cifake/test'),
})

print(f'\n📋 Active databases: {list(DATABASES.keys())}')

In [ ]:
# =================== CELL 4: DATA LOADING HELPERS ===================

def load_dataset_images(dataset_path, max_images=None):
    valid_ext = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}
    image_paths, labels, skipped = [], [], 0

    for folder in sorted(os.listdir(dataset_path)):
        fpath = os.path.join(dataset_path, folder)
        if not os.path.isdir(fpath):
            continue
        fu = folder.upper()
        if fu == 'FAKE':
            label = 1
        elif fu == 'REAL':
            label = 0
        else:
            print(f'  ⚠️  Unknown folder "{folder}" — skipped')
            continue

        collected = []
        for root, _, files in os.walk(fpath):
            for fname in files:
                if os.path.splitext(fname)[1].lower() not in valid_ext:
                    skipped += 1
                    continue
                collected.append(os.path.join(root, fname))
        if max_images:
            collected = collected[:max_images]
        image_paths.extend(collected)
        labels.extend([label] * len(collected))

    if skipped:
        print(f'  ⚠️  {skipped} non-image files skipped')
    return np.array(image_paths), np.array(labels)


def load_db_split(db_name, split='train'):
    paths, labels = load_dataset_images(DATABASES[db_name][split])
    n_real = int(np.sum(labels == 0))
    n_fake = int(np.sum(labels == 1))
    print(f'    {db_name}/{split}: {len(paths):,} images  (REAL={n_real:,}, FAKE={n_fake:,})')
    return paths, labels


def compute_class_weights(labels):
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(labels)
    weights = compute_class_weight('balanced', classes=classes, y=labels)
    return {int(c): float(w) for c, w in zip(classes, weights)}


print('✅ Data loading helpers ready')

In [ ]:
# =================== CELL 5: SRM FILTER BANK ===================
# 8 handcrafted noise-sensitive filters (5 × 5×5 + 3 padded 3×3).
# Identical to the main notebook — copied here so this notebook is self-contained.

filters_5x5 = [
    [[0, 0, -1, 0, 0], [0, -1, 2, -1, 0], [-1, 2, 4, 2, -1], [0, -1, 2, -1, 0], [0, 0, -1, 0, 0]],
    [[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2], [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]],
    [[2, -1, 0, -1, 2], [-1, -2, 3, -2, -1], [0, 3, 0, 3, 0], [-1, -2, 3, -2, -1], [2, -1, 0, -1, 2]],
    [[0, 0, 0, 0, 0], [1, -2, 1, -2, 1], [0, 0, 0, 0, 0], [-1, 2, -1, 2, -1], [0, 0, 0, 0, 0]],
    [[1, -4, 6, -4, 1], [-4, 16, -24, 16, -4], [6, -24, 36, -24, 6], [-4, 16, -24, 16, -4], [1, -4, 6, -4, 1]],
]
filters_3x3_raw = [
    [[0, -1, 0], [-1, 4, -1], [0, -1, 0]],
    [[-1, 2, -1], [2, -4, 2], [-1, 2, -1]],
    [[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]],
]


def _pad_3x3_to_5x5(kernels):
    return [np.pad(k, ((1, 1), (1, 1)), mode='constant', constant_values=0) for k in kernels]


all_filters = np.array(filters_5x5 + _pad_3x3_to_5x5(filters_3x3_raw), dtype=np.float32)

# TF kernel: (5, 5, 1, 8)
srm_filters_tf = tf.constant(
    np.transpose(all_filters[:, :, :, np.newaxis], (1, 2, 3, 0)),
    dtype=tf.float32
)

print(f'✅ SRM filter bank ready: {len(all_filters)} filters, shape {all_filters.shape}')
print(f'   TF kernel shape: {srm_filters_tf.shape}')

In [ ]:
# =================== CELL 6: SRM FILTER FUNCTION & DATASET PIPELINE ===================

AUTOTUNE        = tf.data.AUTOTUNE
SRM_IMG_SIZE    = (256, 256)
SRM_OUTPUT_SHAPE = (256, 256, 24)


def apply_srm_filters_tf(image):
    """
    Apply 8 SRM filters to each RGB channel → tanh normalise → (256,256,24).
    Accepts (H,W,3) or (B,H,W,3). Output is in [0,1].
    """
    squeeze = False
    if len(image.shape) == 3:
        image = image[tf.newaxis, ...]
        squeeze = True

    image = tf.image.resize(image, list(SRM_IMG_SIZE))
    image = tf.cast(image, tf.float32)
    channels = tf.split(image, num_or_size_splits=3, axis=-1)

    feature_maps = []
    for ch in channels:
        fm = tf.nn.conv2d(ch, srm_filters_tf, strides=1, padding='SAME')
        fm = (tf.math.tanh(fm) + 1.0) / 2.0
        feature_maps.append(fm)

    result = tf.concat(feature_maps, axis=-1)
    if squeeze:
        result = result[0]
    return result


def _srm_augment(srm, label):
    srm = tf.image.random_flip_left_right(srm)
    srm = tf.image.random_flip_up_down(srm)
    srm = tf.image.random_brightness(srm, max_delta=0.05)
    srm = tf.clip_by_value(srm, 0.0, 1.0)
    return srm, label


def create_srm_dataset(image_paths, labels, batch_size=32, shuffle=False, augment=False):
    def process(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, list(SRM_IMG_SIZE))
        img = tf.cast(img, tf.float32) / 255.0
        srm = apply_srm_filters_tf(img)
        srm = tf.ensure_shape(srm, list(SRM_OUTPUT_SHAPE))
        return srm, tf.cast(label, tf.float32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(image_paths), 10000), reshuffle_each_iteration=True)
    ds = ds.map(process, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(_srm_augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(1)
    return ds


print('✅ SRM filter function and dataset pipeline ready')
print(f'   Output shape per image: {SRM_OUTPUT_SHAPE}')

In [ ]:
# =================== CELL 7: AUTOENCODER MODEL BUILDER ===================
# SE-Residual autoencoder — identical architecture to the main notebook.
# Input: SRM features (256,256,24) → Latent: (16,16,256) → Reconstruction: (256,256,24)

from tensorflow.keras import layers, Model

REG = tf.keras.regularizers.l2(1e-4)


def _conv_bn_lrelu(x, filters, kernel=3, name=''):
    x = layers.Conv2D(filters, kernel, padding='same', use_bias=False,
                      kernel_regularizer=REG, name=f'{name}_conv')(x)
    x = layers.BatchNormalization(momentum=0.9, epsilon=1e-5, name=f'{name}_bn')(x)
    x = layers.LeakyReLU(negative_slope=0.1, name=f'{name}_act')(x)
    return x


def _se_block(x, ratio=16, name=''):
    ch = x.shape[-1]
    r  = max(1, ch // ratio)
    se = layers.GlobalAveragePooling2D(keepdims=True, name=f'{name}_gap')(x)
    se = layers.Dense(r,  activation='relu',    use_bias=False, name=f'{name}_fc1')(se)
    se = layers.Dense(ch, activation='sigmoid', use_bias=False, name=f'{name}_fc2')(se)
    return layers.Multiply(name=f'{name}_scale')([x, se])


def _res_se_block(x, filters, name=''):
    shortcut = x
    x = _conv_bn_lrelu(x, filters, name=f'{name}_a')
    x = _conv_bn_lrelu(x, filters, name=f'{name}_b')
    x = _se_block(x, ratio=16, name=f'{name}_se')
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same', use_bias=False,
                                 kernel_regularizer=REG, name=f'{name}_proj')(shortcut)
    return layers.Add(name=f'{name}_add')([x, shortcut])


def build_srm_autoencoder():
    """Build SE-Residual SRM autoencoder. Returns (autoencoder, encoder)."""
    inp = layers.Input(shape=(256, 256, 24), name='srm_input')

    # Encoder  256→128→64→32→16
    x = _conv_bn_lrelu(inp, 32, name='enc_s1')
    x = _res_se_block(x, 32, name='enc_res1')
    x = layers.MaxPooling2D((2, 2), name='enc_pool1')(x)         # 128×128

    x = _conv_bn_lrelu(x, 64, name='enc_s2')
    x = _res_se_block(x, 64, name='enc_res2')
    x = layers.MaxPooling2D((2, 2), name='enc_pool2')(x)         # 64×64

    x = _conv_bn_lrelu(x, 128, name='enc_s3')
    x = _res_se_block(x, 128, name='enc_res3')
    x = layers.MaxPooling2D((2, 2), name='enc_pool3')(x)         # 32×32

    x = _conv_bn_lrelu(x, 256, name='enc_s4')
    x = _res_se_block(x, 256, name='enc_res4')
    encoded = layers.MaxPooling2D((2, 2), name='encoded_latent')(x)  # 16×16×256

    # Decoder  16→32→64→128→256
    x = _conv_bn_lrelu(encoded, 256, name='dec_s1')
    x = layers.UpSampling2D((2, 2), name='dec_up1')(x)           # 32×32
    x = _conv_bn_lrelu(x, 128, name='dec_s2')
    x = layers.UpSampling2D((2, 2), name='dec_up2')(x)           # 64×64
    x = _conv_bn_lrelu(x, 64, name='dec_s3')
    x = layers.UpSampling2D((2, 2), name='dec_up3')(x)           # 128×128
    x = _conv_bn_lrelu(x, 32, name='dec_s4')
    x = layers.UpSampling2D((2, 2), name='dec_up4')(x)           # 256×256
    decoded = layers.Conv2D(24, 3, activation='sigmoid', padding='same',
                            name='decoder_output')(x)

    autoencoder = Model(inp, decoded, name='srm_autoencoder_24ch')
    encoder     = Model(inp, encoded, name='srm_encoder_24ch')

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='mse'
    )
    return autoencoder, encoder


print('✅ SRM autoencoder builder ready')
_ae, _enc = build_srm_autoencoder()
print(f'  Autoencoder output: {_ae.output_shape}')
print(f'  Encoder output:     {_enc.output_shape}')
print(f'  Autoencoder params: {_ae.count_params():,}')
del _ae, _enc
tf.keras.backend.clear_session()

In [ ]:
# =================== CELL 8: CNN-SRM CLASSIFIER BUILDER ===================
# Classifier head sits on top of the encoder's (16,16,256) latent space.
# Takes the full SRM input and routes it through: encoder → GAP → Dense head.

def build_cnn_srm(encoder):
    """Build full CNN-SRM model: SRM input (256,256,24) → encoder → classifier head."""
    L2 = tf.keras.regularizers.l2(1e-4)

    cls_input = layers.Input(shape=encoder.output_shape[1:], name='latent_input')
    x = layers.GlobalAveragePooling2D(name='cls_gap')(cls_input)
    x = layers.Dense(256, kernel_regularizer=L2, name='cls_dense1')(x)
    x = layers.BatchNormalization(name='cls_bn1')(x)
    x = layers.Activation('relu', name='cls_act1')(x)
    x = layers.Dropout(0.50, name='cls_drop1')(x)
    x = layers.Dense(128, kernel_regularizer=L2, name='cls_dense2')(x)
    x = layers.BatchNormalization(name='cls_bn2')(x)
    x = layers.Activation('relu', name='cls_act2')(x)
    x = layers.Dropout(0.40, name='cls_drop2')(x)
    x = layers.Dense(64, kernel_regularizer=L2, name='cls_dense3')(x)
    x = layers.Activation('relu', name='cls_act3')(x)
    x = layers.Dropout(0.30, name='cls_drop3')(x)
    output = layers.Dense(1, activation='sigmoid', name='prob_fake')(x)

    cls_head = Model(cls_input, output, name='classifier_head')

    full_input = layers.Input(shape=(256, 256, 24), name='srm_input')
    latent     = encoder(full_input)
    pred       = cls_head(latent)
    model      = Model(full_input, pred, name='SRM_Encoder_Classifier')
    return model


print('✅ CNN-SRM classifier builder ready')

In [ ]:
# =================== CELL 9: EVALUATION HELPERS ===================

from sklearn.metrics import classification_report, roc_auc_score, accuracy_score


def evaluate_model(model, test_ds):
    y_true, y_prob = [], []
    for x_batch, y_batch in test_ds:
        preds = model.predict(x_batch, verbose=0).flatten()
        y_true.extend(y_batch.numpy())
        y_prob.extend(preds)

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob > 0.5).astype(int)

    report = classification_report(
        y_true, y_pred, target_names=['REAL', 'FAKE'], output_dict=True, digits=4
    )
    metrics = {
        'accuracy':       float(accuracy_score(y_true, y_pred)),
        'roc_auc':        float(roc_auc_score(y_true, y_prob)),
        'f1_fake':        float(report['FAKE']['f1-score']),
        'f1_real':        float(report['REAL']['f1-score']),
        'precision_fake': float(report['FAKE']['precision']),
        'recall_fake':    float(report['FAKE']['recall']),
    }
    return metrics, report


def print_eval_report(train_db, test_db, metrics, report):
    tag = '  ✅ [WITHIN-DB]' if train_db == test_db else '  🔀 [CROSS-DB ]'
    print(f'\n{tag}  Trained on {train_db:<15} → Tested on {test_db}')
    sep = '  ' + '─' * 58
    print(sep)
    for cls in ['REAL', 'FAKE']:
        r = report[cls]
        print(f'  {cls:<6}  P={r["precision"]:.4f}  R={r["recall"]:.4f}  F1={r["f1-score"]:.4f}  n={int(r["support"]):,}')
    print(sep)
    print(f'  Accuracy={metrics["accuracy"]:.4f}   ROC-AUC={metrics["roc_auc"]:.4f}')


print('✅ Evaluation helpers ready')

In [ ]:
# =================== CELL 10: MAIN EXPERIMENT LOOP ===================
#
# For each training database:
#   Step A — Train a fresh SRM autoencoder on that database (Option A: no data leakage)
#   Step B — Phase 1: freeze encoder, train classifier head
#   Step C — Phase 2: unfreeze top 30% of encoder, fine-tune end-to-end
#   Step D — Evaluate on ALL 4 databases (within + cross)
#
# TensorBoard logs:
#   cnn_srm/autoencoder_<DB>/  → autoencoder MSE reconstruction loss
#   cnn_srm/train_<DB>/        → classifier accuracy/auc per epoch
#   cnn_srm/cross_db_eval/     → per (train→test) pair evaluation metrics
# ─────────────────────────────────────────────────────────────────────────

BATCH_SIZE  = 32
all_results = {}

for train_db_name in DATABASES:
    print(f'\n{"="*70}')
    print(f'  🎯  TRAINING CNN-SRM ON: {train_db_name}')
    print(f'{"="*70}')

    # ── STEP 0: Load training data ────────────────────────────────────────
    print('\n📦 Loading training data...')
    train_paths, train_lbls = load_db_split(train_db_name, 'train')
    val_paths,   val_lbls   = load_db_split(train_db_name, 'val')
    class_weights           = compute_class_weights(train_lbls)
    print(f'  Class weights: {class_weights}')

    print('\n⚙️  Building SRM datasets...')
    train_srm_ds = create_srm_dataset(
        train_paths, train_lbls, batch_size=BATCH_SIZE, shuffle=True, augment=True)
    val_srm_ds   = create_srm_dataset(
        val_paths, val_lbls, batch_size=BATCH_SIZE, shuffle=False)

    # ── STEP A: Train fresh autoencoder (Option A — no cross-db leakage) ──
    print(f'\n🔧 [STEP A] Training SRM autoencoder on {train_db_name}...')
    gc.collect()
    tf.keras.backend.clear_session()
    autoencoder, encoder = build_srm_autoencoder()

    ae_log      = get_tb_log_dir(train_db_name, suffix='autoencoder')
    encoder_path = os.path.join(MODEL_DIR, f'encoder_{train_db_name}.keras')
    ae_path      = os.path.join(MODEL_DIR, f'autoencoder_{train_db_name}.keras')

    # Autoencoder dataset: input=SRM, target=SRM (reconstruction)
    ae_train_ds = train_srm_ds.map(lambda x, y: (x, x))
    ae_val_ds   = val_srm_ds.map(lambda x, y: (x, x))

    ae_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            ae_path, monitor='val_loss', save_best_only=True, verbose=1),
        tf.keras.callbacks.TensorBoard(
            log_dir=ae_log, histogram_freq=0, update_freq='epoch'),
    ]
    autoencoder.fit(
        ae_train_ds, validation_data=ae_val_ds,
        epochs=20, callbacks=ae_callbacks, verbose=1
    )
    encoder.save(encoder_path)
    print(f'  ✅ Encoder saved → {encoder_path}')

    # ── STEP B: Phase 1 — freeze encoder, train classifier head ──────────
    print(f'\n🚀 [STEP B] Phase 1 — frozen encoder, training classifier head...')
    encoder.trainable = False
    cnn_srm = build_cnn_srm(encoder)
    cnn_srm.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    tb_log   = get_tb_log_dir(train_db_name, suffix='train')
    p1_path  = os.path.join(MODEL_DIR, f'cnn_srm_p1_{train_db_name}.keras')

    p1_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=4,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5,
            patience=2, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            p1_path, monitor='val_auc', mode='max',
            save_best_only=True, verbose=1),
        tf.keras.callbacks.TensorBoard(
            log_dir=tb_log, histogram_freq=0, update_freq='epoch'),
    ]
    cnn_srm.fit(
        train_srm_ds, validation_data=val_srm_ds,
        epochs=15, class_weight=class_weights,
        callbacks=p1_callbacks, verbose=1
    )

    # ── STEP C: Phase 2 — unfreeze top 30% of encoder ────────────────────
    print(f'\n🚀 [STEP C] Phase 2 — unfreeze top 30% of encoder, fine-tune...')
    cnn_srm  = tf.keras.models.load_model(p1_path, compile=False)
    _enc     = cnn_srm.get_layer('srm_encoder_24ch')
    _enc.trainable = True
    fine_tune_from = int(len(_enc.layers) * 0.70)
    for lyr in _enc.layers[:fine_tune_from]:
        lyr.trainable = False
    print(f'  Encoder layers: {len(_enc.layers)} | Frozen: {fine_tune_from} | Trainable: {len(_enc.layers)-fine_tune_from}')

    cnn_srm.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    best_path = os.path.join(MODEL_DIR, f'trained_on_{train_db_name}.keras')
    p2_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=5,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5,
            patience=2, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            best_path, monitor='val_auc', mode='max',
            save_best_only=True, verbose=1),
        tf.keras.callbacks.TensorBoard(
            log_dir=tb_log, histogram_freq=0, update_freq='epoch'),
    ]
    cnn_srm.fit(
        train_srm_ds, validation_data=val_srm_ds,
        epochs=20, class_weight=class_weights,
        callbacks=p2_callbacks, verbose=1
    )

    # ── STEP D: Evaluate on ALL databases ────────────────────────────────
    print(f'\n\n📊 EVALUATION — CNN-SRM trained on {train_db_name}')
    best_model = tf.keras.models.load_model(best_path, compile=False)
    all_results[train_db_name] = {}

    for test_db_name in DATABASES:
        print(f'\n  🔍 Test database: {test_db_name}...')
        test_paths, test_lbls = load_db_split(test_db_name, 'test')
        # Apply SRM preprocessing to the test database images
        test_srm_ds           = create_srm_dataset(
            test_paths, test_lbls, batch_size=BATCH_SIZE, shuffle=False)
        metrics, report       = evaluate_model(best_model, test_srm_ds)
        all_results[train_db_name][test_db_name] = metrics
        log_eval_to_tb(train_db_name, test_db_name, metrics)
        print_eval_report(train_db_name, test_db_name, metrics, report)

    # ── Memory cleanup ────────────────────────────────────────────────────
    del autoencoder, encoder, cnn_srm, best_model, train_srm_ds, val_srm_ds
    gc.collect()
    tf.keras.backend.clear_session()
    print(f'\n✅ {train_db_name} experiment complete — GPU memory cleared.')

results_path = os.path.join(MODEL_DIR, 'all_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f'\n\n{"="*70}')
print('  🏁  ALL EXPERIMENTS COMPLETE')
print(f'{"="*70}')
print(f'\nResults saved → {results_path}')

In [ ]:
# =================== CELL 11: CROSS-DATABASE RESULTS MATRIX ===================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

db_names = list(DATABASES.keys())

for metric_key, metric_label, cmap in [
    ('roc_auc',  'ROC-AUC',  'YlOrRd'),
    ('accuracy', 'Accuracy', 'YlGn'),
    ('f1_fake',  'F1-FAKE',  'Blues'),
]:
    available = [d for d in db_names if d in all_results]
    matrix = [
        [all_results[tr].get(te, {}).get(metric_key, float('nan')) for te in db_names]
        for tr in available
    ]
    df = pd.DataFrame(
        matrix,
        index=[f'Train: {d}' for d in available],
        columns=[f'Test: {d}' for d in db_names]
    )

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.heatmap(
        df.astype(float), annot=True, fmt='.3f',
        cmap=cmap, vmin=0.40, vmax=1.00, ax=ax,
        linewidths=0.5, linecolor='gray',
        cbar_kws={'label': metric_label}
    )
    ax.set_title(
        f'CNN-SRM — Cross-Database {metric_label}\n'
        '(diagonal = within-database · off-diagonal = cross-database)',
        fontsize=13, fontweight='bold'
    )
    ax.set_ylabel('Training Database', fontsize=11)
    ax.set_xlabel('Test Database', fontsize=11)
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig_path = os.path.join(MODEL_DIR, f'cross_db_{metric_key}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {fig_path}')
    print(f'\n{metric_label} Matrix:')
    print(df.to_string())
    print()

In [ ]:
# =================== CELL 12: TENSORBOARD LAUNCH INSTRUCTIONS ===================

print('━' * 60)
print('  📊  TENSORBOARD DASHBOARD')
print('━' * 60)
print()
print('1️⃣  On the SERVER terminal, run:')
print(f'   tensorboard --logdir {TB_LOG_ROOT} --port 6006 --bind_all')
print()
print('2️⃣  On your LOCAL machine, open an SSH tunnel:')
print('   ssh -L 6006:localhost:6006 <your_user>@<server_ip>')
print()
print('3️⃣  Open in browser:  http://localhost:6006')
print()
print('TensorBoard log structure for CNN-SRM:')
print(f'  {TB_LOG_ROOT}/cnn_srm/')
print('  ├── autoencoder_OpenForensics/  ← AE reconstruction loss')
print('  ├── autoencoder_CustomWar/')
print('  ├── autoencoder_CelebDF/')
print('  ├── autoencoder_CiFake/')
print('  ├── train_OpenForensics/        ← classifier accuracy/auc')
print('  ├── train_CustomWar/')
print('  ├── train_CelebDF/')
print('  ├── train_CiFake/')
print('  └── cross_db_eval/')
print('      ├── train_OpenForensics__test_CustomWar/')
print('      └── ...')
print()
print('Tip: all three notebooks (cnn_srm, efficientnet, vit) log to the same')
print(f'     root ({TB_LOG_ROOT}), so you can compare all models in one dashboard.')